<a href="https://colab.research.google.com/github/hmnaz213/native/blob/main/baobab_SIMS_SSOT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 60.7 MB/s eta 0:00:00


In [2]:
import fitz

file_path = '/content/drive/MyDrive/2026/baobab/baobab_personal/SIMS statement of account 15th June 2026.pdf'

def extract_text_from_pdf(path):
    doc = fitz.open(path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

pdf_text = extract_text_from_pdf(file_path)
print("--- Beginning of PDF Content ---")
print(pdf_text[:2000] + "...") # Displaying first 2000 characters to inspect structure

--- Beginning of PDF Content ---
STATEMENT OF INVESTMENTS
 Page : 1
OF
4
Account Number - 524284
Statement Type: Collective Investments Consolidated Statement
Mohammed Nazif Habib
Statement Date: 15-6-2026
P.O.BOX AN 5278
Statement Period: 15-Jun-2026 to 15-Jun-2026
 
Investment Summary
As on: 15-Jun-2026
Fund Name
Total Contribution
(GHS)
Total Withdrawal
(GHS)
Units
Oustanding
Amortized Price Per 
Unit
(GHS)
**Gains since 
inception
(GHS)
MTM Price Per Unit
(GHS)
Amortized Value
(GHS)
MTM Value
(GHS)
STANBIC INCOME FUNDTRUST - 
AMC
1,000.00
 
71.30
16.2466
158.38
16.2466
1,158.38
1,158.38
STANBIC Income Fund Trust
38,387.32
39,952.12
153.80
12.7407
3,497.95
12.5695
1,959.48
1,933.15
Total
39,387.32
39,952.12
3,656.33
3,117.87
3,091.54
 
Amortized Value (AMV) – is the value of the Fund using cost of the securities plus accrued interest, less accrued expenses. The AMV is different from the redemption value. The redemption value of the Fund is based on the mark-to-
market (MTM) prices o

In [ ]:
from google.colab import auth
import gspread
from google.auth import default
import re
import pandas as pd

# Authenticate to Google Sheets
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

print('Authenticated successfully.')

In [ ]:
# Placeholder for your Google Sheet URL
GSHEET_URL = 'PASTE_YOUR_SHEET_URL_HERE'

def parse_investment_data(text):
    # Extracting account holder and date via regex
    name = re.search(r'Statement Type:.*?\n(.*?)\n', text, re.S).group(1).strip()
    acc_num = re.search(r'Account Number - (\d+)', text).group(1)

    # Basic parser for the table rows based on known fund names found in text
    # We extract the specific funds and their values
    data_rows = []
    funds = ["STANBIC INCOME FUNDTRUST - AMC", "STANBIC Income Fund Trust"]

    for fund in funds:
        if fund in text:
            # This regex captures the numeric values following the fund name
            pattern = fund + r'\s+([\d,.]+)\s+([\d,.]*)\s+([\d,.]+)\s+([\d,.]+)\s+([\d,.]+)\s+([\d,.]+)\s+([\d,.]+)\s+([\d,.]+)'
            match = re.search(pattern, text)
            if match:
                row = [name, acc_num, fund] + [m.replace(',', '') for m in match.groups()]
                data_rows.append(row)

    columns = ['Holder', 'Account', 'Fund Name', 'Total Contribution', 'Total Withdrawal', 'Units', 'Amortized Price', 'Gains', 'MTM Price', 'Amortized Value', 'MTM Value']
    return pd.DataFrame(data_rows, columns=columns)

# Prepare DataFrame
df_to_export = parse_investment_data(pdf_text)
display(df_to_export)

# Code to upload (will run once URL is provided)
if 'https' in GSHEET_URL:
    sh = gc.open_by_url(GSHEET_URL)
    worksheet = sh.get_worksheet(0) # Assumes first sheet
    worksheet.update([df_to_export.columns.values.tolist()] + df_to_export.values.tolist())
    print('Data successfully uploaded to Google Sheets!')
else:
    print('Please update the GSHEET_URL variable with your actual sheet URL.')